In [1]:
import numpy as np
from umap import UMAP
import plotly.express as px

/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
Dimensionality reduction visualization of saved feature vectors.
Three algorithms: UMAP, t-SNE, PaCMAP — same interface for easy comparison.
 
UMAP   — fast, good local + global structure, tunable
t-SNE  — classic, excellent local clusters, global structure not meaningful
PaCMAP — 2021, explicitly balances local and global structure preservation
 
Requirements:
    pip install umap-learn pacmap scikit-learn plotly numpy
"""
 
import numpy as np
from umap import UMAP
from sklearn.manifold import TSNE
import pacmap
import plotly.express as px
 
FEATURES_DIR = "./features"
 
 
# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------
 
def _load(model, dataset):
    data     = np.load(f"{FEATURES_DIR}/{model}__{dataset}.npz", allow_pickle=True)
    features = data["features"]
    labels   = data["labels"]
    print(f"{model} × {dataset}  —  {features.shape[0]} samples, "
          f"{len(np.unique(labels))} classes, {features.shape[1]}D features")
    return features, labels
 
 
def _show(embedding, labels, title):
    fig = px.scatter(
        x       = embedding[:, 0],
        y       = embedding[:, 1],
        color   = labels,
        opacity = 0.7,
        title   = title,
        labels  = {"x": "dim 1", "y": "dim 2", "color": "class"},
        width   = 850,
        height  = 650,
    )
    fig.update_traces(marker=dict(size=4))
    fig.show()
 
 
# ---------------------------------------------------------------------------
# UMAP
# n_neighbors: size of local neighbourhood — low = fine local detail,
#              high = more global structure preserved
# min_dist:    how tightly points cluster — low = tight clumps,
#              high = more even spread
# ---------------------------------------------------------------------------
def plot_umap(model, dataset, n_neighbors=15, min_dist=0.1):
    features, labels = _load(model, dataset)
    embedding = UMAP(
        n_neighbors  = n_neighbors,
        min_dist     = min_dist,
        random_state = 42,
        n_components = 2,
    ).fit_transform(features)
    _show(embedding, labels,
          f"UMAP  |  {model} × {dataset}  "
          f"(n_neighbors={n_neighbors}, min_dist={min_dist})")
 
 
# ---------------------------------------------------------------------------
# t-SNE
# perplexity: roughly analogous to n_neighbors — typical range 5–50.
#             low = fine local structure, high = broader neighbourhoods
# Note: inter-cluster distances in t-SNE are NOT meaningful — only
#       within-cluster structure is reliable.
# ---------------------------------------------------------------------------
def plot_tsne(model, dataset, perplexity=30):
    features, labels = _load(model, dataset)
    embedding = TSNE(
        n_components = 2,
        perplexity   = perplexity,
        random_state = 42,
        n_jobs       = -1,
    ).fit_transform(features)
    _show(embedding, labels,
          f"t-SNE  |  {model} × {dataset}  (perplexity={perplexity})")
 
 
# ---------------------------------------------------------------------------
# PaCMAP  (Pairwise Controlled Manifold Approximation, Wang et al. 2021)
# Explicitly separates local and global structure into two loss terms,
# making inter-cluster distances more trustworthy than t-SNE.
# n_neighbors: local neighbourhood size
# MN_ratio:    weight of mid-near pairs (global structure) — default 0.5
# FP_ratio:    weight of further pairs (spread) — default 2.0
# ---------------------------------------------------------------------------
def plot_pacmap(model, dataset, n_neighbors=10, MN_ratio=0.5, FP_ratio=2.0):
    features, labels = _load(model, dataset)
    reducer   = pacmap.PaCMAP(
        n_components = 2,
        n_neighbors  = n_neighbors,
        MN_ratio     = MN_ratio,
        FP_ratio     = FP_ratio,
        random_state = 42,
    )
    embedding = reducer.fit_transform(features)
    _show(embedding, labels,
          f"PaCMAP  |  {model} × {dataset}  "
          f"(n_neighbors={n_neighbors}, MN={MN_ratio}, FP={FP_ratio})")


In [3]:
# ---------------------------------------------------------------------------
# In-distribution / high contamination — expect clean, well-separated clusters
# ---------------------------------------------------------------------------

# ResNet on cifar10: all 10 classes are ImageNet concepts, network trained to separate them
plot_umap("resnet50", "cifar10", n_neighbors=15, min_dist=0.1)

# ResNet on mini_imagenet: 100 classes that are literally ImageNet classes
plot_umap("resnet50", "mini_imagenet", n_neighbors=15, min_dist=0.1)

# DINOv2 on oxford_pets: pet photos abundant in its web training data
plot_umap("dinov2_b", "oxford_pets", n_neighbors=15, min_dist=0.1)

# CLIP on food101: food photography is everywhere in web-scraped image-text data
plot_umap("clip_vit_b32", "food101", n_neighbors=15, min_dist=0.1)

resnet50 × cifar10  —  5000 samples, 10 classes, 2048D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


resnet50 × mini_imagenet  —  5000 samples, 100 classes, 2048D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


dinov2_b × oxford_pets  —  3669 samples, 37 classes, 768D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


clip_vit_b32 × food101  —  5000 samples, 101 classes, 768D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [4]:
# ---------------------------------------------------------------------------
# Truly OOD — does structure emerge from a domain the model never saw?
# ---------------------------------------------------------------------------

# ResNet on chest X-rays: supervised on natural photos, never seen medical imaging
plot_umap("resnet50", "chest_xray", n_neighbors=15, min_dist=0.1)

resnet50 × chest_xray  —  624 samples, 2 classes, 2048D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
# ResNet on satellite imagery: top-down land-use, completely different visual domain
plot_umap("resnet50", "eurosat", n_neighbors=15, min_dist=0.01)

resnet50 × eurosat  —  5000 samples, 10 classes, 2048D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [6]:
# ResNet on histopathology: microscopy slides, cellular scale
plot_umap("resnet50", "pathmnist", n_neighbors=10, min_dist=0.5)
plot_tsne("resnet50", "pathmnist", perplexity=30)
plot_pacmap("resnet50", "pathmnist")

resnet50 × pathmnist  —  5000 samples, 9 classes, 2048D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


resnet50 × pathmnist  —  5000 samples, 9 classes, 2048D features


resnet50 × pathmnist  —  5000 samples, 9 classes, 2048D features


In [7]:
# DINOv2 on chest X-rays: self-supervised, no label bias — does it do better than ResNet?
plot_umap("dinov2_b", "chest_xray", n_neighbors=15, min_dist=0.1)

dinov2_b × chest_xray  —  624 samples, 2 classes, 768D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [8]:
# CLIP on pathmnist: web-scale training — does it help with microscopy?
plot_umap("clip_vit_b32", "pathmnist", n_neighbors=15, min_dist=0.1)

clip_vit_b32 × pathmnist  —  5000 samples, 9 classes, 768D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [9]:
# DINOv2 on satellite: visual structure without label supervision
plot_umap("dinov2_b", "eurosat", n_neighbors=15, min_dist=0.1)

dinov2_b × eurosat  —  5000 samples, 10 classes, 768D features


/home/bhed/EVA_DS/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
import numpy as np
from umap import UMAP
import plotly.express as px

"""
Dimensionality reduction visualization of saved feature vectors.
Three algorithms: UMAP, t-SNE, PaCMAP — same interface for easy comparison.
 
UMAP   — fast, good local + global structure, tunable
t-SNE  — classic, excellent local clusters, global structure not meaningful
PaCMAP — 2021, explicitly balances local and global structure preservation
 
Requirements:
    pip install umap-learn pacmap scikit-learn plotly numpy
"""
 
import numpy as np
from umap import UMAP
from sklearn.manifold import TSNE
import pacmap
import plotly.express as px
 
FEATURES_DIR = "./features"
 
 
# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------
 
def _load(model, dataset):
    data     = np.load(f"{FEATURES_DIR}/{model}__{dataset}.npz", allow_pickle=True)
    features = data["features"]
    labels   = data["labels"]
    print(f"{model} × {dataset}  —  {features.shape[0]} samples, "
          f"{len(np.unique(labels))} classes, {features.shape[1]}D features")
    return features, labels
 
 
def _show(embedding, labels, title):
    fig = px.scatter(
        x       = embedding[:, 0],
        y       = embedding[:, 1],
        color   = labels,
        opacity = 0.7,
        title   = title,
        labels  = {"x": "dim 1", "y": "dim 2", "color": "class"},
        width   = 850,
        height  = 650,
    )
    fig.update_traces(marker=dict(size=4))
    fig.show()
 
 
# ---------------------------------------------------------------------------
# UMAP
# n_neighbors: size of local neighbourhood — low = fine local detail,
#              high = more global structure preserved
# min_dist:    how tightly points cluster — low = tight clumps,
#              high = more even spread
# ---------------------------------------------------------------------------
def plot_umap(model, dataset, n_neighbors=15, min_dist=0.1):
    features, labels = _load(model, dataset)
    embedding = UMAP(
        n_neighbors  = n_neighbors,
        min_dist     = min_dist,
        random_state = 42,
        n_components = 2,
    ).fit_transform(features)
    _show(embedding, labels,
          f"UMAP  |  {model} × {dataset}  "
          f"(n_neighbors={n_neighbors}, min_dist={min_dist})")
 
 
# ---------------------------------------------------------------------------
# t-SNE
# perplexity: roughly analogous to n_neighbors — typical range 5–50.
#             low = fine local structure, high = broader neighbourhoods
# Note: inter-cluster distances in t-SNE are NOT meaningful — only
#       within-cluster structure is reliable.
# ---------------------------------------------------------------------------
def plot_tsne(model, dataset, perplexity=30):
    features, labels = _load(model, dataset)
    embedding = TSNE(
        n_components = 2,
        perplexity   = perplexity,
        random_state = 42,
        n_jobs       = -1,
    ).fit_transform(features)
    _show(embedding, labels,
          f"t-SNE  |  {model} × {dataset}  (perplexity={perplexity})")
 
 
# ---------------------------------------------------------------------------
# PaCMAP  (Pairwise Controlled Manifold Approximation, Wang et al. 2021)
# Explicitly separates local and global structure into two loss terms,
# making inter-cluster distances more trustworthy than t-SNE.
# n_neighbors: local neighbourhood size
# MN_ratio:    weight of mid-near pairs (global structure) — default 0.5
# FP_ratio:    weight of further pairs (spread) — default 2.0
# ---------------------------------------------------------------------------
def plot_pacmap(model, dataset, n_neighbors=10, MN_ratio=0.5, FP_ratio=2.0):
    features, labels = _load(model, dataset)
    reducer   = pacmap.PaCMAP(
        n_components = 2,
        n_neighbors  = n_neighbors,
        MN_ratio     = MN_ratio,
        FP_ratio     = FP_ratio,
        random_state = 42,
    )
    embedding = reducer.fit_transform(features)
    _show(embedding, labels,
          f"PaCMAP  |  {model} × {dataset}  "
          f"(n_neighbors={n_neighbors}, MN={MN_ratio}, FP={FP_ratio})")
    

# ResNet on satellite imagery: top-down land-use, completely different visual domain
plot_umap("resnet50", "eurosat", n_neighbors=15, min_dist=0.01)
